# PAP — Capítulo 4 — Lógica de Agentes
## Hands-On — Etapa 1: investigando um agente de manutenção

**Ideia central:** a LLM pode solicitar o próximo passo, mas o Python continua responsável por validar, autorizar e executar as ações.

Nesta etapa você receberá o agente **pronto e funcional**. O objetivo não é reescrever o código. Execute as células em ordem, acompanhe os logs e localize onde cada responsabilidade aparece.

### Antes de começar

1. Abra este notebook no **Google Colab**.
2. No painel **Secrets**, crie um segredo chamado `GOOGLE_API_KEY`.
3. Cole sua chave da Gemini API e permita o acesso do notebook ao Secret.
4. Execute as células na ordem.
5. Nos testes, observe principalmente os logs e a lista `acoes_executadas`.

> Não coloque a chave diretamente em uma célula do notebook.

## 1. Preparação do ambiente

Vamos usar as mesmas bibliotecas e a mesma linha de trabalho do capítulo: `google-genai`, Interactions API, Pydantic, Function Calling e logs.

In [ ]:
%pip install -q -U "google-genai>=2.3.0,<3" "pydantic>=2,<3"

In [ ]:
from google import genai
from google.colab import userdata
from pydantic import BaseModel, ConfigDict, Field, ValidationError

import json
import logging

api_key = userdata.get("GOOGLE_API_KEY")

if not api_key:
    raise ValueError(
        "Adicione GOOGLE_API_KEY aos Secrets do Google Colab antes de executar."
    )

MODELO = "gemini-3.6-flash"

client = genai.Client(api_key=api_key)

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)

print("Ambiente preparado.")

## 2. Cenário da central de manutenção

A aplicação mantém um pequeno cadastro de equipamentos e alguns chamados em memória.

O cenário inicial foi preparado para produzir comportamentos diferentes:

- `LAB-02` existe e não possui chamado aberto;
- `LAB-03` já está em manutenção e possui um chamado aberto;
- `PC-17` existe e não possui chamado aberto.

In [ ]:
def criar_dados_iniciais():
    equipamentos = {
        "LAB-02": {
            "tipo": "projetor",
            "local": "Laboratório 2",
            "status": "em operação"
        },
        "LAB-03": {
            "tipo": "projetor",
            "local": "Laboratório 3",
            "status": "em manutenção"
        },
        "PC-17": {
            "tipo": "computador",
            "local": "Laboratório 1",
            "status": "em operação"
        }
    }

    chamados = [
        {
            "id": "CH-001",
            "codigo": "LAB-03",
            "descricao": "Projetor sem imagem",
            "status": "aberto"
        }
    ]

    return equipamentos, chamados


equipamentos, chamados = criar_dados_iniciais()

print("Equipamentos cadastrados:", list(equipamentos.keys()))
print("Chamados iniciais:", chamados)

## 3. Capacidades reais da aplicação

As funções abaixo são Python comum. Elas não dependem da LLM.

A LLM poderá **solicitar** o uso dessas capacidades quando elas forem declaradas como tools, mas a execução continuará acontecendo aqui no programa.

In [ ]:
def consultar_equipamento(codigo):
    equipamento = equipamentos.get(codigo)

    if equipamento is None:
        return {
            "erro": "equipamento não encontrado",
            "codigo": codigo
        }

    return {
        "codigo": codigo,
        **equipamento
    }


def consultar_chamado(codigo):
    for chamado in chamados:
        if (
            chamado["codigo"] == codigo
            and chamado["status"] == "aberto"
        ):
            return {
                "existe": True,
                "chamado": chamado
            }

    return {
        "existe": False,
        "codigo": codigo
    }


def registrar_chamado(codigo, descricao):
    chamado = {
        "id": f"CH-{len(chamados) + 1:03d}",
        "codigo": codigo,
        "descricao": descricao,
        "status": "aberto"
    }

    chamados.append(chamado)
    return chamado

## 4. Declaração das tools

As funções continuam locais. As estruturas abaixo apenas descrevem para a LLM quais capacidades existem e quais argumentos são esperados.

Neste notebook o modelo deve solicitar **uma tool por vez**. O resultado volta para a LLM antes da próxima decisão.

In [ ]:
consultar_equipamento_tool = {
    "type": "function",
    "name": "consultar_equipamento",
    "description": (
        "Consulta um equipamento pelo código e retorna tipo, local e status. "
        "Use esta tool antes de afirmar informações cadastrais do equipamento."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "codigo": {
                "type": "string",
                "description": "Código do equipamento, como LAB-02"
            }
        },
        "required": ["codigo"]
    }
}


consultar_chamado_tool = {
    "type": "function",
    "name": "consultar_chamado",
    "description": (
        "Verifica se existe um chamado aberto para o equipamento informado. "
        "Não cria nem altera chamados."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "codigo": {
                "type": "string",
                "description": "Código do equipamento"
            }
        },
        "required": ["codigo"]
    }
}


registrar_chamado_tool = {
    "type": "function",
    "name": "registrar_chamado",
    "description": (
        "Registra um novo chamado de manutenção. "
        "Só deve ser solicitada depois de confirmar que o equipamento existe "
        "e que não há chamado aberto para ele."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "codigo": {
                "type": "string",
                "description": "Código do equipamento"
            },
            "descricao": {
                "type": "string",
                "description": "Descrição objetiva do problema"
            }
        },
        "required": ["codigo", "descricao"]
    }
}


tools = [
    consultar_equipamento_tool,
    consultar_chamado_tool,
    registrar_chamado_tool
]

print("Tools disponíveis:", [tool["name"] for tool in tools])

## 5. Contratos, allowlist e autorização

Aqui aparecem três controles diferentes:

1. **Pydantic** valida os argumentos.
2. A **allowlist** determina quais funções podem ser executadas.
3. A autorização verifica se o perfil atual pode executar a ação.

Além disso, uma regra de negócio impede a abertura de chamado para equipamento inexistente ou para um equipamento que já possua chamado aberto.

In [ ]:
class ArgumentosConsulta(BaseModel):
    model_config = ConfigDict(
        strict=True,
        extra="forbid"
    )

    codigo: str = Field(min_length=1)


class ArgumentosChamado(BaseModel):
    model_config = ConfigDict(
        strict=True,
        extra="forbid"
    )

    codigo: str = Field(min_length=1)
    descricao: str = Field(min_length=5)


tools_permitidas = {
    "consultar_equipamento": (
        consultar_equipamento,
        ArgumentosConsulta
    ),
    "consultar_chamado": (
        consultar_chamado,
        ArgumentosConsulta
    ),
    "registrar_chamado": (
        registrar_chamado,
        ArgumentosChamado
    )
}


acoes_por_perfil = {
    "consulta": [
        "consultar_equipamento",
        "consultar_chamado"
    ],
    "tecnico": [
        "consultar_equipamento",
        "consultar_chamado",
        "registrar_chamado"
    ]
}


def autorizar_acao(perfil, acao):
    permitidas = acoes_por_perfil.get(perfil, [])
    return acao in permitidas


def validar_regra_negocio(nome, argumentos):
    if nome != "registrar_chamado":
        return None

    codigo = argumentos["codigo"]

    if codigo not in equipamentos:
        return {
            "erro": "regra_negocio",
            "detalhe": "não é permitido abrir chamado para equipamento inexistente"
        }

    for chamado in chamados:
        if (
            chamado["codigo"] == codigo
            and chamado["status"] == "aberto"
        ):
            return {
                "erro": "regra_negocio",
                "detalhe": (
                    f"já existe um chamado aberto para {codigo}: "
                    f"{chamado['id']}"
                )
            }

    return None

### Executor controlado

Repare que o nome vindo da LLM não é transformado diretamente em código.

O programa procura a tool na allowlist, valida os argumentos, verifica autorização e regras de negócio e só então chama a função Python.

In [ ]:
def executar_tool(nome, argumentos, perfil):
    configuracao = tools_permitidas.get(nome)

    if configuracao is None:
        logging.warning("Tool recusada: %s", nome)
        return {
            "erro": "tool_nao_permitida",
            "detalhe": f"A tool '{nome}' não está disponível."
        }

    funcao, modelo_argumentos = configuracao

    try:
        dados = modelo_argumentos.model_validate(argumentos)

    except ValidationError as erro:
        logging.warning(
            "Argumentos inválidos para %s: %s",
            nome,
            erro.errors()
        )
        return {
            "erro": "argumentos_invalidos",
            "detalhe": erro.errors()
        }

    argumentos_validados = dados.model_dump()

    if not autorizar_acao(perfil, nome):
        logging.warning(
            "Ação não autorizada para o perfil %s: %s",
            perfil,
            nome
        )
        return {
            "erro": "acao_nao_autorizada",
            "detalhe": (
                f"O perfil '{perfil}' não pode executar '{nome}'."
            )
        }

    recusa = validar_regra_negocio(
        nome,
        argumentos_validados
    )

    if recusa is not None:
        logging.warning(
            "Ação recusada por regra de negócio: %s",
            recusa["detalhe"]
        )
        return recusa

    try:
        logging.info("Executando tool: %s", nome)

        return funcao(
            **argumentos_validados
        )

    except Exception as erro:
        logging.error(
            "Falha durante a execução da tool %s: %s",
            nome,
            erro
        )
        return {
            "erro": "falha_execucao",
            "detalhe": "A tool falhou durante a execução."
        }

## 6. Ciclo do agente

Agora as peças são reunidas.

O modelo recebe a solicitação e as tools. Se solicitar uma função, o Python executa o fluxo de controle e devolve um `function_result`. Esse resultado passa a fazer parte das informações disponíveis para a próxima decisão.

O ciclo termina quando a LLM produz uma resposta final ou quando o limite máximo de iterações é atingido.

In [ ]:
INSTRUCAO_SISTEMA = """
Você é o agente da central de manutenção da CampusTech.

Use somente as tools disponibilizadas para consultar ou alterar dados do sistema.
Nunca invente status de equipamento, chamados, identificadores ou resultados de tools.

Quando o usuário pedir abertura de chamado:
1. consulte o equipamento;
2. se ele existir, consulte se há chamado aberto;
3. só depois solicite registrar_chamado, se realmente for necessário.

Se uma tool retornar erro, recusa ou falta de autorização, considere esse resultado
antes de responder. Nunca diga que uma operação foi realizada se a tool não a confirmou.

Solicite no máximo uma tool por interação.
Quando já houver informação suficiente, responda ao usuário sem solicitar nova tool.
"""

MAX_ITERACOES = 5


def encontrar_function_call(interaction):
    for passo in interaction.steps or []:
        if passo.type == "function_call":
            return passo

    return None

In [ ]:
def executar_agente(solicitacao, perfil="tecnico"):
    estado = {
        "solicitacao": solicitacao,
        "perfil": perfil,
        "iteracoes": 0,
        "acoes_executadas": [],
        "resposta_final": None
    }

    logging.info("Nova solicitação")
    logging.info("Perfil: %s", perfil)

    interaction = client.interactions.create(
        model=MODELO,
        system_instruction=INSTRUCAO_SISTEMA,
        input=solicitacao,
        tools=tools
    )

    concluido = False

    while (
        estado["iteracoes"] < MAX_ITERACOES
        and not concluido
    ):
        estado["iteracoes"] += 1

        logging.info(
            "Iteração %s",
            estado["iteracoes"]
        )

        chamada = encontrar_function_call(interaction)

        if chamada is None:
            resposta = (interaction.output_text or "").strip()

            if not resposta:
                resposta = (
                    "O fluxo terminou sem uma resposta textual."
                )

            estado["resposta_final"] = resposta
            concluido = True
            break

        logging.info(
            "Tool solicitada: %s",
            chamada.name
        )

        logging.info(
            "Argumentos recebidos: %s",
            chamada.arguments
        )

        resultado_tool = executar_tool(
            chamada.name,
            chamada.arguments,
            perfil
        )

        estado["acoes_executadas"].append({
            "tool": chamada.name,
            "argumentos": chamada.arguments,
            "resultado": resultado_tool
        })

        logging.info(
            "Resultado observado: %s",
            resultado_tool
        )

        interaction = client.interactions.create(
            model=MODELO,
            input=[
                {
                    "type": "function_result",
                    "name": chamada.name,
                    "call_id": chamada.id,
                    "result": [
                        {
                            "type": "text",
                            "text": json.dumps(
                                resultado_tool,
                                ensure_ascii=False
                            )
                        }
                    ]
                }
            ],
            tools=tools,
            previous_interaction_id=interaction.id
        )

    if not concluido:
        logging.warning(
            "Fluxo interrompido pelo limite de iterações."
        )

        estado["resposta_final"] = (
            "Não foi possível concluir o atendimento dentro "
            "do limite de execução."
        )

    return estado

## 7. Testes

Execute os casos abaixo **na ordem**. Não altere o agente ainda.

As respostas textuais da LLM podem variar. O que deve ser investigado é o caminho percorrido: tools solicitadas, validações, autorizações, resultados e condição de parada.

### Teste 1 — consulta simples

O perfil `consulta` pode ler informações, mas não pode registrar chamados.

In [ ]:
resultado_1 = executar_agente(
    "Qual é a situação do LAB-03?",
    perfil="consulta"
)

print("\nRESPOSTA FINAL")
print(resultado_1["resposta_final"])

print("\nAÇÕES OBSERVADAS")
for item in resultado_1["acoes_executadas"]:
    print("-", item["tool"], "->", item["resultado"])

### Teste 2 — decisões sucessivas

Aqui uma única tool não basta. O agente precisa obter novas informações antes de decidir se deve registrar o chamado.

In [ ]:
resultado_2 = executar_agente(
    (
        "O projetor LAB-02 está desligando sozinho. "
        "Verifique se já existe um chamado e, se não existir, abra um."
    ),
    perfil="tecnico"
)

print("\nRESPOSTA FINAL")
print(resultado_2["resposta_final"])

print("\nAÇÕES OBSERVADAS")
for item in resultado_2["acoes_executadas"]:
    print("-", item["tool"], "->", item["resultado"])

### Confira a alteração de estado

Se o registro foi autorizado e executado, a lista `chamados` agora contém uma nova ocorrência. Essa mudança foi realizada pela função Python, não pela LLM.

In [ ]:
print("Chamados atuais:")

for chamado in chamados:
    print(chamado)

### Teste 3 — ação conhecida, mas não autorizada

`PC-17` existe e não possui chamado aberto. O pedido faz sentido, mas o perfil `consulta` não pode executar `registrar_chamado`.

A LLM pode solicitar a operação. O Python é quem deve recusá-la.

In [ ]:
resultado_3 = executar_agente(
    (
        "O computador PC-17 não liga. "
        "Verifique a situação e abra um chamado se ainda não existir."
    ),
    perfil="consulta"
)

print("\nRESPOSTA FINAL")
print(resultado_3["resposta_final"])

print("\nAÇÕES OBSERVADAS")
for item in resultado_3["acoes_executadas"]:
    print("-", item["tool"], "->", item["resultado"])

### Teste 4 — equipamento inexistente

O agente não deve inventar informações para `LAB-99` e não pode abrir um chamado para um equipamento que não existe no cadastro.

In [ ]:
resultado_4 = executar_agente(
    (
        "O projetor LAB-99 está com problema. "
        "Verifique e abra um chamado para ele."
    ),
    perfil="tecnico"
)

print("\nRESPOSTA FINAL")
print(resultado_4["resposta_final"])

print("\nAÇÕES OBSERVADAS")
for item in resultado_4["acoes_executadas"]:
    print("-", item["tool"], "->", item["resultado"])

## 8. Investigação final

Sem modificar o código, localize no notebook:

- onde a **LLM escolhe** solicitar uma tool;
- onde o programa verifica se a tool está na **allowlist**;
- onde os argumentos são validados com **Pydantic**;
- onde ocorre a **autorização por perfil**;
- onde uma regra de negócio confronta a decisão com o **estado atual**;
- qual linha realmente executa a **função Python**;
- como o resultado retorna à LLM por `function_result`;
- onde o programa reconhece a **resposta final**;
- onde o limite de iterações impede que o ciclo continue indefinidamente.

### Pergunta para levar à Etapa 2

> Se todo o conteúdo relacionado a equipamentos e chamados fosse removido, quais partes desta arquitetura poderiam ser reaproveitadas para construir um agente em outro domínio?

Na Etapa 2, você usará essa resposta para adaptar a arquitetura ao caso **PetTech**.

## Opcional — reiniciar o cenário

Ao executar novamente os testes, lembre-se de que `registrar_chamado()` altera a lista em memória.

Use a célula abaixo quando quiser restaurar os dados iniciais.

In [ ]:
equipamentos, chamados = criar_dados_iniciais()

print("Cenário reiniciado.")
print("Chamados:", chamados)